> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Propagating uncertainty through functions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/main/docs/user_guide/03_functions_and_lifting.ipynb)

Write a function for one value of each input, then use it with values, distributions, or batches.
This chapter explains how those choices change the result and its dependence on other inputs.
The savings calculation is a small example of the calling rules.
The last section declares the inputs and the output that a function accepts.
For a full forecasting analysis, follow [tutorial 4](../tutorials/04_forecasting.ipynb).

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab's preinstalled PyMC 5 does not work with ArviZ 1.x, so the install upgrades PyMC.
    %pip install --quiet "probpipe-core[pymc] @ git+https://github.com/TARPS-group/prob-pipe.git@main"

In [2]:
import jax
import jax.numpy as jnp

from probpipe import (
    ApplicabilityError,
    EmpiricalDistribution,
    Normal,
    NumericArray,
    NumericArrayBatch,
    NumericArraySpec,
    Record,
    config,
    expectation,
    function,
    mean,
    positive,
    pushforward,
    quantile,
    variance,
    workflow_run,
)

## A function of values

Our running example is a savings account.
`future_value` computes the balance after `years` years of a deposit of `principal` that grows at the annual `rate`.
The `@function` decorator makes it a ProbPipe-native function, which is an instance of `Function`:

In [3]:
@function
def future_value(principal: float, rate: jax.Array, years: float) -> jax.Array:
    return principal * (1.0 + rate) ** years


balance = future_value(1000.0, 0.05, 10.0)
print("the function:", future_value)
print("balance of 1000 after 10 years at 5%:", balance.raw)

the function: future_value(principal, rate, years)
balance of 1000 after 10 years at 5%: 1628.8947


`print` shows the function's notation, which is its label followed by its parameters.
Called with numbers, the function evaluates its body once.
It returns the balance as a `NumericArray`, and its `.raw` is the underlying JAX array.
The type hints state what each parameter accepts and what the body returns, and ProbPipe reads them when it decides how to apply the function.

## Calling a function on a distribution

The rate of return is uncertain.
Suppose it follows a normal distribution with mean 5% and standard deviation 2%.
We pass that distribution in place of the number:

In [4]:
rate = Normal("rate", 0.05, 0.02)

with workflow_run(seed=0):
    balance = future_value(1000.0, rate, 10.0)

print("result:", balance)
print("mean balance after 10 years:", mean(balance).raw)
print("90% interval of the balance:", quantile(balance, jnp.array([0.05, 0.95])).raw)
print("route:", balance.provenance.metadata["route"])

result: future_value(1000.0, rate ~ Normal, 10.0)
mean balance after 10 years: 1637.634


90% interval of the balance: [1158.4106 2165.6436]
route: sampling_lift


The result is the distribution of the balance.
It displays as `future_value(1000.0, rate ~ Normal, 10.0)`, which is the function applied to a draw of `rate` from the distribution labeled `Normal`.
ProbPipe *lifts* the function to the distribution: by default it draws 256 rates, evaluates the body at each draw, and returns an `EmpiricalDistribution` over the 256 balances.
The result is a distribution like any other, so `mean` and `quantile` apply to it.
Its provenance stores the evaluation-rule name `sampling_lift` under the key `route` and records that the result is approximate.
The `workflow_run` block seeds the draws, so running the cell again reproduces the result.

More draws generally reduce Monte Carlo error, although an individual estimate can move farther from its target.
`using(max_evaluations=...)` returns a copy of the function that uses another number of draws, and `config.sampling.max_evaluations` sets the evaluation limit for lifted functions that set none of their own.
The separate setting `config.sampling.num_samples` controls Monte Carlo summaries, such as an approximate `mean`:

In [5]:
with workflow_run(seed=0):
    balance_4000 = future_value.using(max_evaluations=4000)(1000.0, rate, 10.0)
print("draws set by using:", balance_4000.num_atoms)

config.sampling.max_evaluations = 1000
with workflow_run(seed=0):
    balance_1000 = future_value(1000.0, rate, 10.0)
print("draws with the default set to 1000:", balance_1000.num_atoms)

config.sampling.reset()

draws set by using: 4000
draws with the default set to 1000: 1000


An `EmpiricalDistribution` is a distribution over a finite set of values called **atoms**, so a function can be evaluated at every atom instead of at random draws.
Here the rate takes one of three values with equal probability:

In [6]:
three_rates = EmpiricalDistribution(jnp.array([0.02, 0.05, 0.08]), output_name="rate")
balance_3 = future_value(1000.0, three_rates, 10.0)

print("balance at each rate:", balance_3.atoms.raw)
print("route:", balance_3.provenance.metadata["route"])
print("exact:", balance_3.provenance.metadata["exact"])

balance at each rate: [1218.9941 1628.8938 2158.9258]
route: empirical_enumeration
exact: True


The evaluation rule `empirical_enumeration` evaluates the body once per atom and keeps each atom's weight, so the result is the exact distribution of the balance.
It needs no seed, since it draws nothing.
ProbPipe enumerates the atoms when their number is at most `max_evaluations`, 256 by default, and it samples from them otherwise.

## Values and distributions together

A call can pass a value to some parameters and a distribution to others, as the calls above did.
Each distribution argument is lifted, and each value is passed to every evaluation unchanged.
When two arguments are distributions, their draws are dependent if the arguments come from one distribution and independent otherwise.
`gap` makes the difference visible:

In [7]:
@function
def gap(a: jax.Array, b: jax.Array) -> jax.Array:
    return a - b


other_rate = Normal("other_rate", 0.05, 0.02)
with workflow_run(seed=0):
    same = gap(rate, rate)
    independent = gap(rate, other_rate)

print("variance of rate - rate:", variance(same).raw)
print("variance of rate - other_rate:", variance(independent).raw)

variance of rate - rate: 0.0
variance of rate - other_rate: 0.00078610674


Both arguments of `gap(rate, rate)` come from one distribution, so each evaluation reads them from the same draw, and the difference is always 0.
The arguments of `gap(rate, other_rate)` come from two distributions, which are drawn independently, so the variance of the difference is near $2 \times 0.02^2 = 0.0008$.
The same rule keeps the dependence between two fields of one joint distribution, such as a posterior, and [Building a model](05_building_a_model.ipynb) describes those fields.
A copy that `with_label` returns is the same distribution under a new label, so `gap(rate, rate.with_label("r"))` also reads both arguments from one draw.
To draw two independent values, we construct the distribution twice, as `other_rate` does.

ProbPipe passes the distribution itself to a parameter whose type hint is `Distribution`, so the body can call operations on it.

## Batches of inputs

A **batch** holds several values of one kind along named groups of axes called **levels**.
Each element of a batch is a separate input, so a function called on a batch evaluates its body once per element.
Here three horizons form a batch on the level `horizon`:

In [8]:
horizons = NumericArrayBatch(jnp.array([5.0, 10.0, 20.0]), "horizon", label="years")
print("levels of the batch:", dict(zip(horizons.level_names, horizons.batch_shape)))
print("the horizons:", horizons.raw)
print("the second horizon:", horizons[1].raw)

levels of the batch: {'horizon': 3}
the horizons: [ 5. 10. 20.]
the second horizon: 10.0


A function called on a batch evaluates its body at each element and returns a batch of the results on the same level:

In [9]:
balances = future_value(1000.0, 0.05, horizons)
print("result:", type(balances).__name__)
print("balance at 5%, by horizon:", balances.raw)

result: NumericArrayBatch
balance at 5%, by horizon: [1276.2812 1628.8938 2653.2952]


Two batches on different levels form a grid of every combination, whereas two batches on the same level are paired element by element.
Within one level, sizes follow NumPy's broadcasting rule, so a batch of size one on a level is paired with every element of the other batch.
These levels count separate evaluations, whereas the entries of one ordinary array belong to one input value.
A batch of two rates on the level `scenario` therefore gives a 2 × 3 grid of balances:

In [10]:
rates = NumericArrayBatch(jnp.array([0.03, 0.05]), "scenario", label="rate")
grid = future_value(1000.0, rates, horizons)
print("levels of the grid:", dict(zip(grid.level_names, grid.batch_shape)))
print("balance by scenario (rows) and horizon (columns):")
print(grid.raw)

levels of the grid: {'scenario': 2, 'horizon': 3}
balance by scenario (rows) and horizon (columns):
[[1159.2738 1343.916  1806.1102]
 [1276.2812 1628.8938 2653.2952]]


A batch and a distribution can be passed in one call.
The result is a batch of distributions, one per horizon, and the operations applied to it return a batch of results:

In [11]:
with workflow_run(seed=1):
    balance_by_horizon = future_value(1000.0, rate, horizons)

print("result:", type(balance_by_horizon).__name__)
print("mean balance by horizon:", mean(balance_by_horizon).raw)
print("distribution at the first horizon:", balance_by_horizon[0].label)

result: DistributionBatch


mean balance by horizon: [1286.059  1678.1405 2964.3757]
distribution at the first horizon: future_value[horizon=0]


Indexing the batch by position returns the distribution of one element, labeled by its position on the level.
Each element is computed from its own draws of the rate.

A function can also return a batch, such as the balance at the end of each of ten years.
Called on the distribution of the rate, it returns the distribution of the whole sequence of balances, and each draw is one batch on the level `year`:

In [12]:
@function
def balance_path(principal: float, rate: jax.Array) -> NumericArrayBatch:
    years = jnp.arange(1.0, 11.0)
    return NumericArrayBatch(principal * (1.0 + rate) ** years, "year", label="balance")


with workflow_run(seed=0):
    paths = balance_path(1000.0, rate)
path_means = mean(paths)

print("levels of the atoms:", dict(zip(paths.atoms.level_names, paths.atoms.batch_shape)))
print("levels of the mean:", dict(zip(path_means.level_names, path_means.batch_shape)))
print("mean balance after years 1, 5, and 10:", path_means.raw[jnp.array([0, 4, 9])])

levels of the atoms: {'balance_path': 256, 'year': 10}
levels of the mean: {'year': 10}
mean balance after years 1, 5, and 10: [1048.9791 1274.3679 1637.634 ]


The atoms are 256 sequences of 10 balances, and their first level is named `balance_path` after the function's output.
The mean of a distribution of batches is taken element by element, so it is a batch on the level `year`.
A later function called on `paths` receives each path as its raw array, unless its parameter is annotated with a batch class, such as `NumericArrayBatch`:

In [13]:
@function
def last_year_gain(path: jax.Array) -> jax.Array:
    return path[-1] - path[-2]


with workflow_run(seed=1):
    gains = last_year_gain(paths)

print("mean gain in the tenth year:", mean(gains).raw)

mean gain in the tenth year: 81.15051


The body of `last_year_gain` indexes the raw array of each path, so it is plain JAX code.

## Several outputs as a record

A function that computes several quantities returns them as a dictionary, with the return type `dict[str, jax.Array]`.
ProbPipe returns the dictionary as a record whose fields are the dictionary's keys:

In [14]:
@function
def growth(principal: float, rate: jax.Array, years: float) -> dict[str, jax.Array]:
    balance = principal * (1.0 + rate) ** years
    return {"balance": balance, "gain": balance - principal}


at_5_percent = growth(1000.0, 0.05, 10.0)
print("result:", repr(at_5_percent))
print("gain at 5%:", at_5_percent["gain"].raw)

result: NumericRecord('growth(1000.0, 0.05, 10.0)')
  balance   1628.8947
  gain      628.89465
gain at 5%: 628.89465


Lifted to the distribution of the rate, the function returns a joint distribution of the two fields, and indexing it by a field name gives that field's distribution:

In [15]:
with workflow_run(seed=0):
    outcome = growth(1000.0, rate, 10.0)

print("result:", outcome)
print("mean gain after 10 years:", mean(outcome["gain"]).raw)

result: growth(1000.0, rate ~ Normal, 10.0)
mean gain after 10 years: 637.6341


Both fields of each atom come from the same draw of the rate, so their dependence is kept.

A returned distribution takes the function's label by default.
A returned value displays the application in its label, as `growth(1000.0, 0.05, 10.0)` does above.
The `output_label` argument of `@function` sets another label, which the result of every call takes, so the result carries the name of the quantity it holds:

In [16]:
@function(output_label="savings")
def savings_after(principal: float, rate: jax.Array, years: float) -> jax.Array:
    return principal * (1.0 + rate) ** years


with workflow_run(seed=0):
    savings = savings_after(1000.0, rate, 10.0)
print("the function:", repr(savings_after))
print("label of the result:", savings.label)

the function: Function('savings_after', parameters=('principal', 'rate', 'years'), output_label='savings')
label of the result: savings


## `pushforward`, `expectation`, and `check`

The operation `pushforward(d, f)` returns the distribution of `f(X)` for `X` drawn from the distribution `d`.
It accepts a plain Python callable as well as a `Function`, so it is useful when you want to transform a distribution without decorating the callable.
It takes the distribution first, and the function receives one draw of `d` at a time, even at a parameter annotated `Distribution`, to which the direct call `f(d)` passes the whole distribution.
A function of several parameters takes the others by name in `fixed_args`, so that each draw binds the one parameter left open:

In [17]:
with workflow_run(seed=0):
    balance = pushforward(rate, future_value, fixed_args={"principal": 1000.0, "years": 10.0})

prov_meta = balance.provenance.metadata
print("mean balance after 10 years:", mean(balance).raw)
print(f"provenance: route={prov_meta['route']}, method={prov_meta['method']}")

mean balance after 10 years: 1649.6346
provenance: route=evaluation_rules, method=sampling_lift


`pushforward` selects an evaluation rule as the direct call does, here the sampling lift.
Each fixed argument is a constant, and `pushforward` refuses a distribution or a batch in `fixed_args`.
Its mean differs a little from the first section's, since a workflow derives each call's draws from the seed and the call, and the two calls differ.


In [18]:
def annual_interest(rate: jax.Array) -> jax.Array:
    return 1000.0 * rate


with workflow_run(seed=0):
    interest = pushforward(rate, annual_interest)
print("mean annual interest:", mean(interest).raw)

mean annual interest: 49.80269


`annual_interest` is an ordinary Python function.
`pushforward` wraps it for this computation and returns a distribution of interest values.

The operation `expectation(d, f)` computes the expectation of `f(X)` for `X` drawn from `d`, which is `mean(pushforward(d, f))`.
The function `f` receives one draw of `d` at a time.
An indicator equals 1 when an event occurs and 0 otherwise, so its expectation is the event's probability.
We use an indicator to estimate the probability that the balance doubles in 10 years:

In [19]:
def doubles(balance: jax.Array) -> jax.Array:
    return (balance >= 2000.0).astype(jnp.float32)


with workflow_run(seed=2):
    balance = future_value.using(max_evaluations=4000)(1000.0, rate, 10.0)
probability = expectation(balance, doubles)

print("probability that the balance doubles in 10 years:", probability.raw)
print("route:", probability.provenance.metadata["route"])

probability that the balance doubles in 10 years: 0.13125001
route: closed_form


The balance is an empirical distribution, so `expectation` computes the exact expectation over its 4,000 atoms, and the route `closed_form` records that.
This calculation still approximates the probability under the original normal rate model.
The balance doubles when the rate is at least $2^{1/10} - 1 \approx 7.2\%$, which has probability 0.14 under the normal distribution of the rate.
`expectation` also accepts a distribution with no closed-form expectation, such as `rate` itself, and it then takes the route `pushforward_mean`, which builds the pushforward from draws and takes its mean.

Every ProbPipe-native function has a `check` method, which reports how a call would be computed without running it:

In [20]:
report = future_value.check(1000.0, rate, 10.0)
print("selected rule:", report.selected.method_name)
print("exact:", report.selected.exact)
print("lifted parameters:", report.lifted)

report = future_value.check(1000.0, three_rates, 10.0)
print("selected rule for three rates:", report.selected.method_name)

selected rule: sampling_lift
exact: False
lifted parameters: ('rate',)
selected rule for three rates: empirical_enumeration


The call on `rate` would take the sampling lift over the parameter `rate`, which is approximate, and the call on the three rates would enumerate their atoms.
[Tracking and reproducing results](09_how_results_are_computed.ipynb) explains routes, options, and provenance for every operation.

## Specs

A **spec** describes the kind and structure that a term satisfies.
Use one when you need an explicit declaration for data or a function.

Each term exposes its spec through `.spec`.
Each kind has its own spec class:

- `NumericArraySpec`: the shape, the dtype, and the support of an array;
- `RecordSpec`: the spec of each field of a record, and `NumericRecordSpec` when every field is an array;
- `OpaqueSpec`: the Python type of a non-numeric value;
- `DistributionSpec`: the spec of the distribution itself; its `event_spec` is an `OutputSpec` that declares one draw;
- `FunctionSpec`: the declarations of a function's inputs and of its output;
- `BatchSpec`: the spec of each element of a batch and the size of each level.

In [21]:
weights = NumericArray(jnp.array([0.2, 0.5, 0.3]), label="weights")
site = Record({"count": jnp.array([3, 5, 2]), "area": 1.5}, label="site")
theta = Normal("theta", 0.0, 1.0)


@function
def rescale(x: jax.Array) -> jax.Array:
    # Divide by the total, so the entries sum to one.
    return x / jnp.sum(x)


print("spec of weights:", weights.spec)
print("spec of site:", site.spec)
print("declaration of one draw of theta:", theta.event_spec)
print("spec of rescale:", rescale.spec)

spec of weights: NumericArraySpec(shape=(3,), dtype=float32)
spec of site: NumericRecordSpec(count=(3,), area=())
declaration of one draw of theta: OutputSpec(theta=NumericArraySpec(shape=(), dtype=float32, support=real))
spec of rescale: FunctionSpec()


A draw of `theta` is a scalar under the output `theta`, and its support `real` is the set of values a draw can take.
`rescale` declares neither its inputs nor its output, so its spec is empty, and a call accepts the arguments that its Python body accepts.

Construction checks the value against a spec passed as `spec=`.
Here the spec declares three positive weights, and the array holds two:

In [22]:
three_positive_weights = NumericArraySpec((3,), support=positive)

try:
    NumericArray(jnp.array([0.4, 0.6]), spec=three_positive_weights, label="weights")
except ValueError as error:
    print("construction raised:", error)

construction raised: NumericArray 'weights': value of shape (2,) and dtype float32 does not match spec NumericArraySpec(shape=(3,), support=positive)


Construction checks the shape and whether the stored dtype can be cast to the declared dtype under NumPy's `same_kind` rule.
It preserves the stored array, so `.raw.dtype` and `.dtype` can differ from `.spec.dtype`.
The support is checked by the operations that depend on it, such as `convert` in [Working with results](08_conversion_and_updating.ipynb), which by default refuses a family whose support differs from the distribution's.

A spec can name a dimension in place of a fixed size.
Each value binds the name to its own size, and a name used twice takes one size, so this spec declares a square matrix of any size:

In [23]:
square = NumericArraySpec(("n", "n"))

print("named dimensions:", square.free_dims)
print("a 2 x 2 array matches:", square.is_valid(jnp.eye(2)))
print("a 3 x 3 array matches:", square.is_valid(jnp.eye(3)))
print("a 2 x 3 array matches:", square.is_valid(jnp.ones((2, 3))))
print("the spec with n bound to 4:", square.with_dim_sizes(n=4))

named dimensions: frozenset({'n'})


a 2 x 2 array matches: True


a 3 x 3 array matches: True
a 2 x 3 array matches: False
the spec with n bound to 4: NumericArraySpec(shape=(4, 4))


A function can declare its inputs and its output.
`@function` takes `input_spec`, which maps each parameter to its spec, and `output_spec`, which declares the result.
A call then checks its arguments against the input spec and raises `ApplicabilityError` for an argument that does not match:

In [24]:
@function(input_spec={"counts": NumericArraySpec(("n",))}, output_spec=NumericArraySpec(()))
def total(counts: jax.Array) -> jax.Array:
    return jnp.sum(counts)


print("spec of total:", total.spec)
print("total of three counts:", total(jnp.array([3, 5, 2])).raw)

try:
    total(jnp.ones((2, 3)))
except ApplicabilityError as error:
    print("a matrix of counts raised:", error)

spec of total: FunctionSpec(
    input_spec=InputSpec(counts=NumericArraySpec(shape=('n',))),
    output_spec=OutputSpec(total=NumericArraySpec(shape=())),
)
total of three counts: 10
a matrix of counts raised: Function 'total' input/counts has rank 2, expected rank 1 from shape ('n',)


The output spec names the output `total`, and a distribution's `event_spec` names the outputs of a draw in the same way.
A function's result and a distribution's draw therefore share one kind of spec, the output spec, which [Building a model](05_building_a_model.ipynb) uses to merge a model from its parts.

## See also

- [Functions](../api/functions.md): `Function`, `function`, and `CallReport`.
- [Operations](../api/operations.md): `pushforward`, `expectation`, and the moments such as `mean`.
- [Values and records](../api/values.md): `NumericArrayBatch` and the other batches.
- [Input and output specs](../api/input_and_output_specs.md): the spec classes and the supports, such as `positive`.
- [Workflows and reproducibility](../api/workflows.md): `workflow_run` and `config.sampling`.
- The chapters [Uncertainty in a computation](01_uncertainty_in_a_computation.ipynb), [Your data: values, records, and batches](04_values_and_records.ipynb), [Distributions](02_distributions.ipynb), [Building a model](05_building_a_model.ipynb), and [Tracking and reproducing results](09_how_results_are_computed.ipynb).
- [Tutorial 2](../tutorials/02_fitting.ipynb) applies a function to a prior and a posterior, and [tutorial 4](../tutorials/04_forecasting.ipynb) forecasts with a lifted function and compares harvest scenarios as a batch.